# FEP pipeline

Runs the FEP pipeline, locally or remotely.

The relationship of a specific system and pipeline stages: `pipeline -> pair -> replica (a state, b state)`

- `pipeline` is the main process that runs and analyzes replicas
- `pair` is a specific pair of ligands whose difference in activity needs to be calculated
- `replica` - calculation of a specific pair with random parameters for generating water, ions, etc. for statistical analysis of the results, the name consists of the name of the pair and **replica** (calculation option): 'l00_l01_r1'
- `state` - the system of complex A and b; ligands a and B

These parts of the pipeline form a hierarchical structure.

---

1. **Pipeline training**

   All processes (parameterization, box creation...) they run in parallel for each protein, ligand, etc.
   - Parameterization
   - Ligand hybridization map
   - Ligand hybridization

2. **replica**

   Each replica is a separate unit, the states of which are calculated independently of the others.
   - Creation of states: complexA, complexB, ligandA, ligandB
   - Create water boxes for each condition
   - Energy minimization of states
   - Equilibrium
   - Trajectory split into snapshots
   - Nonequilibrium transitions A<->B for snapshots
   - BAR analysis

3. **Pipeline training**

   Connecting the analysis data of each replicas together
   - Analysis of the transition energies for each pair, statistical analysis of the calculation error


In [1]:
from pathlib import Path
from md import pipelines


In [2]:
pipeline = pipelines.FEP(
    workdir=Path("temp_FEP"),
    raw_prot_path=Path("examples/data/FEP/3l9h_prepared-dry.pdb"),
    raw_ligs_sdf=Path("examples/data/FEP/ligands.sdf"),
    extra_sdf_paths=[],
    pairs=[("CHEMBL1078774", "CHEMBL1089056")],
    workers=2,
    compress=False,
    # Fast
    n_replicas=1,
    emtol=10000,
    eq_time=0.002,
    internal_steps=4,
    n_frames=3,
    neq_time=0.001,
    discard_time=0.001,
)

In [3]:
pipeline.run()

17:34 |INFO   |   0|   PIPELINE RUN
17:34 |INFO   |   1|   PARAMETRIZE PROTEIN
17:34 |INFO   |   1| V PARAMETRIZE PROTEIN
17:34 |INFO   |   5|   PARAMETRIZE EXTRAS
17:34 |INFO   |   5| V PARAMETRIZE EXTRAS
17:34 |INFO   |   6|   GEN GRAPH
17:34 |INFO   |   6| V GEN GRAPH
17:34 |INFO   |   7|   PARAMETRIZE LIGANDS
17:34 |INFO   |   7| V PARAMETRIZE LIGANDS
17:34 |INFO   |  10|   CREATE HYBRIDS
17:34 |INFO   |  10| V CREATE HYBRIDS
17:34 |INFO   |  17|   CREATE REPLICAS
17:34 |INFO   |  17| V CREATE REPLICAS
17:34 |INFO   |  18|   RUN REPLICAS
17:34 |INFO   |  19|   PIPELINE REPLICA RUN
17:34 |INFO   |  20|   Replica l00_l22
17:41 |INFO   |  20| V Replica l00_l22
17:41 |INFO   | 117| Progress:   1/1 fails:  0
17:41 |INFO   |  19| V PIPELINE REPLICA RUN
17:41 |INFO   |  18| V RUN REPLICAS
17:41 |INFO   | 118|   ANALYZE
17:41 |INFO   | 118| V ANALYZE
17:41 |INFO   |   0| V PIPELINE RUN


,ligand_a,ligand_b,ddG,ddG_error,ddG_berror
0,CHEMBL1078774,CHEMBL1089056,-6.75,0 2.948847 dtype: float64,6.328262
